# Evaluation of the material calibration

**Purpose.** This notebook measures how well the calibrated neural material predicts the
reference signal received power (RSRP) of minimization-of-drive-tests (MDT) records that the
calibration did not use. It compares the calibrated material with two reference models: the
materials of the scene file and the neural material before the calibration. The method of
the calibration is "Neural Materials" of Hoydis et al., *Learning Radio Environments by
Differentiable Ray Tracing*, [arXiv:2311.18558](https://arxiv.org/abs/2311.18558).

**Inputs.**

| File | Content |
|---|---|
| `configs/config.yaml` | The configuration. `src.utils.load_config` composes it with the other files of `configs/`. The notebook reads `seed`, the Sionna RT configuration (`simulation`), the network structure (`calibration.network`), and the configuration of the evaluation (`evaluation`). |
| `data/raw/cells.csv` | One row for each cell |
| `data/processed/train.parquet` | The train records. They give the power scale of the reference models. |
| `data/processed/test.parquet` | The test records. They give the error of each model. |
| `outputs/calibration/calibration.npz` | The calibrated network and its power scale, from `notebooks/03_material_calibration.ipynb` |

**Method.** `src.simulation.compute_rsrp` simulates the RSRP of each model. The functions of
`src.evaluation` calculate the error statistics. Section 2 gives the equations.

**Figures.** `configure_matplotlib` sets the figure style. Each figure is saved as a PNG file
in `reports/figures/`. The text below each figure is its caption.

**Outputs.** Each table is saved as a CSV file in `reports/tables/`, with the prefix
`evaluation_`. The notebook writes no other file.

**Requirements.** Run `notebooks/03_material_calibration.ipynb` first. The notebook needs an
NVIDIA GPU. It runs the path solver three times on the train records and on the test
records.

## 1. Setup

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from omegaconf import OmegaConf

from src.calibration import load_calibration, received_powers, scale_estimate
from src.core import read_cells
from src.data import mdt_to_arrays
from src.evaluation import (
    error_statistics,
    grouped_statistics,
    nearest_distance,
    pair_errors,
    paired_bootstrap,
)
from src.simulation import (
    apply_neural_material,
    assign_material,
    building_height_map,
    compute_rsrp,
    load_rt_scene,
    make_material_network,
)
from src.utils import (
    COLORS,
    DATA_DIR,
    FIGURE_WIDTH_DOUBLE,
    OUTPUTS_DIR,
    load_config,
    save_figure,
    save_table,
    setup_notebook,
)

RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
CALIBRATION_DIR = OUTPUTS_DIR / "calibration"
PARTS = ("train", "test")
ITU, INITIAL, CALIBRATED = "ITU materials", "Initial neural material", "Calibrated neural material"
MODELS = (ITU, INITIAL, CALIBRATED)
cfg = load_config()
device = setup_notebook(cfg)

In [ ]:
def label_panels(axes):
    "Put the labels (a), (b), ... above the top left corner of each panel."
    for index, ax in enumerate(np.ravel(axes)):
        ax.set_title(f"({chr(ord('a') + index)})", loc="left", fontweight="bold")


def format_map(ax):
    "Give a map panel an equal scale on the two axes and axis labels in meters."
    ax.set(xlabel="$x$ (m)", ylabel="$y$ (m)", aspect="equal")
    ax.grid(False)


def ecdf(values):
    "Return the sorted values and the empirical cumulative distribution function."
    ordered = np.sort(np.asarray(values))
    return ordered, np.arange(1, ordered.size + 1) / ordered.size


def statistic_labels(quantiles):
    "Return the display name of each statistic of `error_statistics`."
    labels = {
        "pairs": "Pairs",
        "no_path_fraction": "No path (fraction)",
        "bias_db": "Bias (dB)",
        "mae_db": "MAE (dB)",
        "rmse_db": "RMSE (dB)",
        "std_db": "Std (dB)",
    }
    for quantile in quantiles:
        labels[f"q{100.0 * quantile:g}_abs_db"] = f"Q{100.0 * quantile:g} of |e| (dB)"
    return labels | {"pearson": "PCC", "smape": "SMAPE"}

The configuration of the evaluation is below.

In [ ]:
print(OmegaConf.to_yaml(cfg.evaluation))

## 2. Method

### 2.1 Error of one pair

For one received (record, cell) pair, let $P_\mathrm{dB}$ be the measured RSRP and let
$\hat{P}_\mathrm{dB}$ be the simulated RSRP, the two in dBm. Each model has a power scale
$\alpha$ that multiplies the measured power (arXiv:2311.18558, Section III-F). The signed
error of the pair is

$$e = P_\mathrm{dB} + 10 \log_{10} \alpha - \hat{P}_\mathrm{dB}.$$

A positive $e$ shows that the model gives less power than the scaled measurement.
`src.calibration.log_error` calculates $e$. The absolute value $|e|$ is the absolute
logarithmic error (ALE) of the paper (Eq. 44). A pair with no simulated path has no $e$.

### 2.2 Statistics

For the $N$ pairs that have a simulated path, `src.evaluation.error_statistics` calculates:

| Statistic | Equation | What it shows |
|---|---|---|
| Bias | $\bar{e} = \frac{1}{N}\sum_i e_i$ | A systematic offset of the model |
| MAE | $\frac{1}{N}\sum_i \lvert e_i \rvert$ | The mean ALE of the paper |
| RMSE | $\sqrt{\frac{1}{N}\sum_i e_i^2}$ | The error, with more weight on large errors |
| Std | $\sqrt{\frac{1}{N}\sum_i (e_i - \bar{e})^2}$ | The error after the removal of the bias |
| Quantiles | quantiles `evaluation.quantiles` of $\lvert e \rvert$ | The tail of the error |
| PCC | Pearson correlation of $P_\mathrm{dB}$ and $\hat{P}_\mathrm{dB}$ | The agreement of the spatial pattern, independent of an offset |
| SMAPE | mean of $\lvert r - 1 \rvert / (r + 1)$, $r = 10^{e/10}$ | The calibration loss (Eq. 39). A pair with no path counts as 1. |

The fraction of the received pairs with no simulated path is also a statistic. A model that
loses many paths can have a small MAE on the pairs that stay. Studies of ray-tracer
calibration report the same statistics: bias, standard deviation, RMSE, MAE, high
percentiles, and the correlation (Bakirtzis et al., arXiv:2509.19337; Sun et al.,
Data in Brief 2026, DOI 10.1016/j.dib.2026.113364).

### 2.3 Models

| Model | Materials | Power scale |
|---|---|---|
| ITU materials | The materials of the scene file (Recommendation ITU-R P.2040) | Eq. 43 with all train records |
| Initial neural material | `make_material_network` with `seed`, before the first gradient step | Eq. 43 with all train records |
| Calibrated neural material | The network of `calibration.npz` | The scale of `calibration.npz` |

The scenario generator used the ITU materials to make the data. The ITU materials are thus
the true materials, and only the measurement errors separate them from the records. Their
error is the lowest error that a model can get. The initial neural material is the start
point of the calibration. The calibration is useful if it moves the error from the initial
value to the value of the ITU materials.

### 2.4 Comparison of two models

The two models use the same pairs, so the comparison uses the difference of the absolute
errors of each pair. `src.evaluation.paired_bootstrap` calculates the difference of the
MAE of two models and a percentile confidence interval. The bootstrap samples the records,
not the pairs: the pairs of one record share the error of the reported position, thus they
are not independent. If the interval does not contain 0, the difference is larger than the
variation of the test records.

### 2.5 Generalization in space

`split_records` divides the records at random. A test record can thus be near a train
record, and a model that learns the local environment can have a small test error only for
this reason (Dempsey et al., arXiv:2411.17752). `src.evaluation.nearest_distance` gives the
horizontal distance from each test record to the nearest train record. If the error of the
calibrated material increases with this distance, the calibration does not generalize far
from the train records. The paper also tells that neural materials generalize poorly to
areas with no measurements.

### 2.6 Reproducibility

The ray tracer gives the same RSRP in each run. The calibration does not: each run of
notebook 03 gives different weights. The section "Reproducibility" of the README gives the
cause. A difference between two runs of this notebook can thus come from a new run of
notebook 03. A difference between two models that is smaller than the spread between runs
of notebook 03 is not a reliable result.

## 3. Data

`mdt_to_arrays` gives one position for each record and one RSRP for each (record, cell)
pair. Table 1 gives the size of the parts.

In [ ]:
cells = read_cells(RAW_DIR / "cells.csv")
frames = {name: pd.read_parquet(PROCESSED_DIR / f"{name}.parquet") for name in PARTS}
positions, rsrp_dbm = {}, {}
for name, frame in frames.items():
    positions[name], rsrp_dbm[name] = mdt_to_arrays(frame, cells)

save_table(
    pd.DataFrame(
        {
            "Records": [len(positions[name]) for name in PARTS],
            "Received (record, cell) pairs": [len(frames[name]) for name in PARTS],
        },
        index=list(PARTS),
    ).style.set_caption("Table 1. Size of the parts of the MDT data."),
    "evaluation_parts",
)

`load_calibration` reads the calibrated network and its power scale. The cell below checks
that the network has the structure of `calibration.network`. The initial neural material
uses this structure, thus the comparison is valid only if the two agree.

In [ ]:
calibrated_material, calibrated_scale = load_calibration(
    CALIBRATION_DIR / "calibration.npz", "calibrated"
)
state = calibrated_material.state()
structure = {key: int(state[key]) for key in cfg.calibration.network}
if structure != dict(cfg.calibration.network):
    raise ValueError(
        f"calibration.npz has the network {structure}, but calibration.network is "
        f"{dict(cfg.calibration.network)}. Run notebook 03 again."
    )
print(f"Network of calibration.npz: {structure}")

## 4. Simulation

The cell below simulates the RSRP of each model at the positions of the train and the test
records. One scene gets the materials of the three models in sequence: first the ITU
materials of the scene file, then the initial neural material, then the calibrated
material. Table 2 gives the power scale of each model.

In [ ]:
scene = load_rt_scene(cfg.simulation.scene.name)


def simulate():
    "Return the simulated RSRP (dBm) of each part with the current materials of the scene."
    return {name: compute_rsrp(scene, cells, positions[name], cfg.simulation) for name in PARTS}


simulated = {ITU: simulate()}
apply_neural_material(scene, make_material_network(**cfg.calibration.network, seed=cfg.seed))
simulated[INITIAL] = simulate()
assign_material(scene, calibrated_material)
simulated[CALIBRATED] = simulate()

scales = {
    model: scale_estimate(*received_powers(rsrp_dbm["train"], simulated[model]["train"]))
    for model in (ITU, INITIAL)
}
scales[CALIBRATED] = calibrated_scale

save_table(
    pd.DataFrame(
        {
            "Power scale (dB)": [10.0 * np.log10(scales[model]) for model in MODELS],
            "Source of the scale": ["Eq. 43, train records"] * 2 + ["calibration.npz"],
        },
        index=list(MODELS),
    )
    .style.set_caption("Table 2. Power scale of each model.")
    .format({"Power scale (dB)": "{:.2f}"}),
    "evaluation_models",
)

## 5. Overall error

`pair_errors` gives the error of each received pair. Table 3 gives the statistics of each
model on the train records and on the test records. The difference between the two parts
is the generalization gap. The calibration used the train records, thus a calibrated
material with a much smaller train error than test error has learned the train records,
not the environment.

In [ ]:
quantiles = list(cfg.evaluation.quantiles)
labels = statistic_labels(quantiles)
pairs = {
    (model, name): pair_errors(frames[name], cells, simulated[model][name], scales[model])
    for model in MODELS
    for name in PARTS
}
overall = pd.DataFrame({key: error_statistics(value, quantiles) for key, value in pairs.items()}).T
overall.index.names = ["Model", "Part"]

save_table(
    overall.rename(columns=labels)
    .style.set_caption("Table 3. Error statistics of each model on each part.")
    .format(precision=2)
    .format({labels["pairs"]: "{:.0f}", labels["no_path_fraction"]: "{:.3f}"}),
    "evaluation_overall",
)

## 6. Error distribution

Fig. 1 shows the distribution of the error on the test records. The bias of a model moves
its histogram away from 0. A large tail of $|e|$ shows that a model fails in some areas,
even when its MAE is small.

In [ ]:
test_error = {model: pairs[model, "test"]["error_db"].dropna() for model in MODELS}
edges = np.histogram_bin_edges(np.concatenate(list(test_error.values())), bins="auto")
calibrated_test = pairs[CALIBRATED, "test"].dropna(subset="error_db")
scaled_measured = calibrated_test["rsrp_dbm"] + 10.0 * np.log10(scales[CALIBRATED])

fig, axes = plt.subplots(1, 3, figsize=(FIGURE_WIDTH_DOUBLE, 2.6))
for color, model in zip(COLORS, MODELS, strict=False):
    axes[0].plot(*ecdf(test_error[model].abs()), color=color, label=model)
    axes[1].hist(test_error[model], bins=edges, histtype="step", color=color, label=model)
axes[0].set(xlabel="$|e|$ (dB)", ylabel="Empirical CDF", ylim=(0, 1))
axes[0].legend(loc="lower right", fontsize="x-small")
axes[1].axvline(0.0, color="black", linewidth=0.5)
axes[1].set(xlabel="$e$ (dB)", ylabel="Pairs")
axes[2].scatter(
    scaled_measured, calibrated_test["simulated_dbm"], s=3, linewidths=0, color=COLORS[2]
)
limits = [
    min(scaled_measured.min(), calibrated_test["simulated_dbm"].min()),
    max(scaled_measured.max(), calibrated_test["simulated_dbm"].max()),
]
axes[2].plot(limits, limits, color="black", linewidth=0.5)
axes[2].set(
    xlabel=r"$P_\mathrm{dB} + 10 \log_{10} \alpha$ (dBm)",
    ylabel=r"$\hat{P}_\mathrm{dB}$ (dBm)",
    aspect="equal",
)
label_panels(axes)
save_figure(fig, "evaluation_error_distribution")
plt.show()

**Fig. 1.** Error on the test records. (a) Empirical distribution of the absolute error of
each model. (b) Histogram of the signed error of each model. (c) Simulated RSRP of the
calibrated neural material against the scaled measured RSRP. The line shows equal values.

## 7. Error by cell and by distance

The error can depend on the cell and on the distance from the cell. Table 4 gives the MAE
of each model for each cell. Table 5 gives the statistics in `evaluation.num_bins` distance
bands with equal numbers of pairs. Fig. 2 shows the two tables.

In [ ]:
test_pairs = {model: pairs[model, "test"] for model in MODELS}
by_cell = pd.DataFrame(
    {
        model: grouped_statistics(frame, frame["cell"], quantiles)["mae_db"]
        for model, frame in test_pairs.items()
    }
)
by_cell.insert(0, "Pairs", test_pairs[CALIBRATED].groupby("cell").size())
save_table(
    by_cell.style.set_caption("Table 4. MAE (dB) of each model on the test pairs of each cell.")
    .format(precision=2)
    .format({"Pairs": "{:.0f}"}),
    "evaluation_by_cell",
)

distance = test_pairs[CALIBRATED]["distance_m"]
distance_band = pd.qcut(distance, cfg.evaluation.num_bins, duplicates="drop").rename(
    "Distance band (m)"
)
by_distance = pd.concat(
    {
        model: grouped_statistics(frame, distance_band, quantiles)[["pairs", "bias_db", "mae_db"]]
        for model, frame in test_pairs.items()
    },
    axis=1,
)
save_table(
    by_distance.rename(columns=labels, level=1)
    .style.set_caption("Table 5. Error statistics of each model in each distance band.")
    .format(precision=2),
    "evaluation_by_distance",
)

In [ ]:
band_distance = distance.groupby(distance_band, observed=True).median()

fig, (ax_cell, ax_distance) = plt.subplots(1, 2, figsize=(FIGURE_WIDTH_DOUBLE, 2.6))
by_cell[list(MODELS)].plot.bar(ax=ax_cell, color=COLORS[: len(MODELS)], width=0.8, legend=False)
ax_cell.set(xlabel="Cell", ylabel="MAE (dB)")
ax_cell.tick_params(axis="x", labelrotation=90)
for color, model in zip(COLORS, MODELS, strict=False):
    ax_distance.plot(
        band_distance.to_numpy(),
        by_distance[model]["mae_db"].to_numpy(),
        "o-",
        color=color,
        markersize=3,
        label=model,
    )
ax_distance.set(xlabel="Median distance to the cell in the band (m)", ylabel="MAE (dB)")
ax_distance.legend(fontsize="x-small")
label_panels([ax_cell, ax_distance])
save_figure(fig, "evaluation_error_structure")
plt.show()

**Fig. 2.** Structure of the error on the test records. (a) MAE of each model for each cell.
(b) MAE of each model in each distance band. The points are at the median distance of the
band.

## 8. Generalization in space

Table 6 gives the statistics of the test pairs in `evaluation.num_bins` bands of the
distance to the nearest train record. Fig. 3 shows the MAE in each band and the mean
absolute error of each test record on the map.

In [ ]:
neighbor = pd.Series(
    nearest_distance(test_pairs[CALIBRATED][["x", "y"]].to_numpy(), positions["train"]),
    index=test_pairs[CALIBRATED].index,
)
neighbor_band = pd.qcut(neighbor, cfg.evaluation.num_bins, duplicates="drop").rename(
    "Distance to the nearest train record (m)"
)
by_neighbor = pd.concat(
    {
        model: grouped_statistics(frame, neighbor_band, quantiles)[["pairs", "bias_db", "mae_db"]]
        for model, frame in test_pairs.items()
    },
    axis=1,
)
save_table(
    by_neighbor.rename(columns=labels, level=1)
    .style.set_caption(
        "Table 6. Error statistics of each model by the distance to the nearest train record."
    )
    .format(precision=2),
    "evaluation_by_neighbor",
)

In [ ]:
height_map = building_height_map(scene, cfg.scenario.grid.tile_size_m)
ny, nx = height_map.heights.shape
map_extent = [
    height_map.x_min,
    height_map.x_min + nx * height_map.tile_size,
    height_map.y_min,
    height_map.y_min + ny * height_map.tile_size,
]
is_building = height_map.heights > cfg.scenario.grid.min_building_height_m
record_error = (
    calibrated_test.assign(abs_error_db=calibrated_test["error_db"].abs())
    .groupby("record")
    .agg(x=("x", "first"), y=("y", "first"), abs_error_db=("abs_error_db", "mean"))
)
neighbor_distance = neighbor.groupby(neighbor_band, observed=True).median()

fig, (ax_band, ax_map) = plt.subplots(
    1, 2, figsize=(FIGURE_WIDTH_DOUBLE, 2.9), width_ratios=[1, 1.4]
)
for color, model in zip(COLORS, MODELS, strict=False):
    ax_band.plot(
        neighbor_distance.to_numpy(),
        by_neighbor[model]["mae_db"].to_numpy(),
        "o-",
        color=color,
        markersize=3,
        label=model,
    )
ax_band.set(xlabel="Median distance to the nearest train record (m)", ylabel="MAE (dB)")
ax_band.legend(fontsize="x-small")
ax_map.imshow(is_building, origin="lower", extent=map_extent, cmap="Greys", alpha=0.3)
ax_map.scatter(*positions["train"][:, :2].T, s=2, color="black", linewidths=0, label="train")
points = ax_map.scatter(
    record_error["x"],
    record_error["y"],
    c=record_error["abs_error_db"],
    s=4,
    linewidths=0,
    vmin=0,
    vmax=record_error["abs_error_db"].quantile(max(quantiles)),
)
ax_map.scatter(
    [cell.x for cell in cells], [cell.y for cell in cells], marker="^", color="black", label="cell"
)
format_map(ax_map)
ax_map.legend(loc="upper right", markerscale=2, fontsize="x-small")
fig.colorbar(points, ax=ax_map, label="Mean $|e|$ of the record (dB)", extend="max")
label_panels([ax_band, ax_map])
save_figure(fig, "evaluation_spatial")
plt.show()

**Fig. 3.** Error as a function of the position. (a) MAE of each model on the test pairs in
each band of the distance to the nearest train record. The points are at the median distance
of the band. (b) Mean absolute error of each test record with the calibrated neural material.
The black points are the train records and the triangles are the sites. The color scale
stops at the highest quantile of `evaluation.quantiles`.

## 9. Comparison of the models

Table 7 gives the difference of the MAE of two models on the test pairs, with the bootstrap
interval of `evaluation.bootstrap`. A negative difference shows that the calibrated
material has the smaller error.

In [ ]:
bootstrap = cfg.evaluation.bootstrap
comparisons = {
    f"{CALIBRATED} minus {reference}": paired_bootstrap(
        test_pairs[reference],
        test_pairs[CALIBRATED],
        bootstrap.num_samples,
        bootstrap.confidence,
        seed=cfg.seed,
    )
    for reference in (INITIAL, ITU)
}
comparison = pd.DataFrame(comparisons, index=["MAE difference (dB)", "Lower (dB)", "Upper (dB)"]).T
comparison["Interval contains 0"] = (comparison["Lower (dB)"] <= 0) & (
    comparison["Upper (dB)"] >= 0
)

save_table(
    comparison.style.set_caption(
        f"Table 7. Difference of the MAE on the test pairs, with the "
        f"{100 * bootstrap.confidence:g} % bootstrap interval."
    ).format(precision=2),
    "evaluation_comparison",
)

## 10. Findings

The cell below prints the results of this run. The notes after it give the limits of the
evaluation. They are valid for each run.

In [ ]:
for model in MODELS:
    test_mae = overall.loc[(model, "test"), "mae_db"]
    train_mae = overall.loc[(model, "train"), "mae_db"]
    print(f"{model}: test MAE {test_mae:.2f} dB, train MAE {train_mae:.2f} dB")
for name, result in comparisons.items():
    print(f"{name}: {result.estimate:+.2f} dB [{result.lower:+.2f}, {result.upper:+.2f}] dB")
near, far = by_neighbor[CALIBRATED]["mae_db"].iloc[[0, -1]]
print(
    f"{CALIBRATED}: MAE {near:.2f} dB in the nearest band and {far:.2f} dB in the farthest "
    "band of the distance to the train records"
)

- **The data is synthetic.** The ITU materials made the data, thus their error comes only
  from the measurement errors of the generator. On real MDT data, no model is the true
  model, and the ITU materials are only a baseline.
- **The power scale of the reference models uses the train records.** The scale of the
  calibrated material comes from the moving average of the calibration (Eq. 42). The scale
  of the two reference models is the least-squares value of Eq. 43 on all train records.
- **A random division does not test the generalization in space.** Section 8 shows the
  effect of the distance to the train records. A division into areas would give a stronger
  test (arXiv:2411.17752).
- **The result changes with each calibration run.** Section 2.6 gives the cause. One run of
  notebook 03 does not show the spread between runs.
- **The data has missing values below the report limit.** A cell below
  `scenario.measurement.min_rsrp_dbm` is not in a record. The pairs of weak cells are thus
  missing, and the statistics do not include them.
- **The reported positions have a GNSS error.** The simulation uses the reported position,
  not the true position. This error stays in each model.